In [1]:
import os
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset

In [2]:
IMAGE_FOLDER = "Filtered_Train"

annotations = pd.read_csv("Filtered_Train/filtered_annotations.csv")

annotations.head()

,filename,width,height,class,xmin,ymin,xmax,ymax
0,848617_jpg.rf.b6167489414b68fcfe2ccda38d24ac4a...,500.0,333.0,ahaetulla prasina,127.0,0.0,375.0,333.0
1,203459385_jpg.rf.73858ce8179c66e61c2203f4966e8...,500.0,375.0,bungarus caeruleus,144.0,48.0,332.0,325.0
2,178020213_jpg.rf.902a97ff37bebf466e6800e389762...,281.0,500.0,ptyas korros,0.0,161.0,281.0,347.0
3,AUG81008965_jpeg.rf.f10a9b8c28dba23257d0ee0594...,374.0,500.0,bungarus caeruleus,142.0,134.0,361.0,436.0
4,32814798_jpeg.rf.10944af1cd359dd50fca3e4518592...,243.0,500.0,ahaetulla prasina,60.0,129.0,179.0,325.0


In [3]:
classes = sorted(annotations["class"].unique()) # sorts them alphabetically.

class_to_idx = { # mapping integers {"cobra": 0, "python": 1, "viper": 2}
    cls: idx
    for idx, cls in enumerate(classes) # enumerate give pairs like (0, "cobra"), (1, "python")
}

idx_to_class = { # reverse {0: "cobra", 1: "python", 2: "viper"}
    idx: cls
    for cls, idx in class_to_idx.items() 
}



In [4]:
print(class_to_idx)

{'ahaetulla prasina': 0, 'amphiesma stolatum': 1, 'bungarus caeruleus': 2, 'bungarus fasciatus': 3, 'chrysopelea ornata': 4, 'daboia russelii': 5, 'dendrelaphis pictus': 6, 'naja naja': 7, 'ophiophagus hannah': 8, 'psammodynastes pulverulentus': 9, 'ptyas korros': 10, 'ptyas mucosa': 11, 'trimeresurus albolabris': 12, 'trimeresurus purpureomaculatus': 13, 'xenochrophis piscator': 14}


In [5]:
# creates a new column called Label
annotations["Label"] = annotations["class"].map(class_to_idx) #  applies the mapping to every value in the class column.

annotations.sample(10)

,filename,width,height,class,xmin,ymin,xmax,ymax,Label
2860,193447067_jpg.rf.2c5c3bd8a8ae8ce880f2c46af9fc0...,302.0,500.0,ophiophagus hannah,42.0,282.0,204.0,412.0,8
1001,4600859_jpg.rf.df36c1e03b249939d97243a3bdfcb9c...,500.0,333.0,ahaetulla prasina,189.0,3.0,381.0,333.0,0
2975,34864366_jpeg.rf.a5e78d8cd0f5fc625434b20a98cf1...,500.0,282.0,naja naja,130.0,14.0,391.0,186.0,7
5582,170807753_jpeg.rf.f80c7813463ea902b2d21cd88c9e...,500.0,375.0,psammodynastes pulverulentus,167.0,0.0,346.0,375.0,9
4918,34029307_jpg.rf.2fa485ab387fa79a87c1df531820e6...,414.0,500.0,daboia russelii,31.0,42.0,248.0,441.0,5
3476,7940258_jpeg.rf.c7976ace7a5ca688819532c4b1b5e6...,500.0,333.0,trimeresurus albolabris,105.0,117.0,500.0,228.0,12
5457,145596180_jpg.rf.c6a51a1fb36659deb1df23bf099bb...,375.0,500.0,trimeresurus albolabris,95.0,185.0,269.0,304.0,12
3376,212015002_jpg.rf.8a6538f56b20ce41d2482f141fb99...,500.0,281.0,xenochrophis piscator,121.0,67.0,272.0,155.0,14
1729,197451467_jpeg.rf.b2fdf81c7314d97b43a76815d966...,500.0,375.0,ptyas korros,157.0,0.0,431.0,375.0,10
3479,63459037_jpeg.rf.96cabc2f32d04e9cadba484495e41...,500.0,333.0,naja naja,21.0,52.0,445.0,284.0,7


In [ ]:
IMAGE_FOLDER = "Filtered_Train"
filtered_species = "Filtered_Train/filtered_annotations.csv"

annotations = pd.read_csv(filtered_species)

classes = sorted(annotations["class"].dropna().astype(str).unique()) 

class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

# Converts the class values to strings and maps them to numeric labels.
annotations["Label"] = annotations["class"].astype(str).map(class_to_idx)

# Save the updated annotations back to the same CSV with the Label column added
annotations.to_csv(filtered_species, index=False)

annotations.head()

,filename,width,height,class,xmin,ymin,xmax,ymax,Label
0,848617_jpg.rf.b6167489414b68fcfe2ccda38d24ac4a...,500.0,333.0,ahaetulla prasina,127.0,0.0,375.0,333.0,0
1,203459385_jpg.rf.73858ce8179c66e61c2203f4966e8...,500.0,375.0,bungarus caeruleus,144.0,48.0,332.0,325.0,2
2,178020213_jpg.rf.902a97ff37bebf466e6800e389762...,281.0,500.0,ptyas korros,0.0,161.0,281.0,347.0,10
3,AUG81008965_jpeg.rf.f10a9b8c28dba23257d0ee0594...,374.0,500.0,bungarus caeruleus,142.0,134.0,361.0,436.0,2
4,32814798_jpeg.rf.10944af1cd359dd50fca3e4518592...,243.0,500.0,ahaetulla prasina,60.0,129.0,179.0,325.0,0


In [ ]:
class SnakeDataset(Dataset):
    def __init__(self, filtered_species, image_folder, transform=None):
        self.data = pd.read_csv(filtered_species)
        self.image_folder = image_folder
        self.transform = transform

        if "Label" not in self.data.columns:
            raise KeyError("CSV must contain a 'Label' column.")

            self.data["Label"] = self.data["class"].astype(str).map(self.class_to_idx)
        else:
            self.class_to_idx = None

    def __len__(self): # Return the number of samples in the dataset
        return len(self.data)

    def __getitem__(self, index): #return for a specific index. for ex: dataset[0]
        row = self.data.iloc[index] #  selects one row by position.
        

        image_path = os.path.join(self.image_folder, row["filename"])
        image = Image.open(image_path).convert("RGB") #  makes sure the image is loaded in standard RGB format.

        label = int(row["Label"])

        if self.transform: # resize image, convert to tensor, normalize pixel values
            image = self.transform(image)

        return image, label

In [19]:
from torchvision import transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)), # changes image size to 224x224.
    transforms.ToTensor(), # converts the image to a PyTorch tensor.
])

In [21]:
dataset = SnakeDataset(
    filtered_species="Filtered_Train/filtered_annotations.csv",
    image_folder="Filtered_Train",
    transform=train_transform
)

In [ ]:
print(len(dataset))
image, label = dataset[96]

print(type(image))
print(image.size())
print(label)

6110
<class 'torch.Tensor'>
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
7


In [25]:
image, label = dataset[0]

print(type(image))
print(image.shape)
print(label)

<class 'torch.Tensor'>
torch.Size([3, 224, 224])
0


In [28]:
DataLoader = torch.utils.data.DataLoader
train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    drop_last=False,
    pin_memory=torch.cuda.is_available()
)

In [29]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])


In [30]:
print(images.dtype)
print(images.min())
print(images.max())

torch.float32
tensor(0.)
tensor(1.)


In [31]:
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)

print(f"Total Images      : {len(dataset)}")
print(f"Total Classes     : {len(class_to_idx)}")
print(f"Batch Size        : {32}")
print(f"Device            : {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print("=" * 50)

DATASET SUMMARY
Total Images      : 6110
Total Classes     : 15
Batch Size        : 32
Device            : CUDA


In [32]:
import json

with open("class_mapping.json", "w") as f:
    json.dump(class_to_idx, f, indent=4)

print("✓ class_mapping.json saved")

✓ class_mapping.json saved
